In [ ]:
import pandas as pd
import numpy as np
import torch

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [ ]:
columns = ["ID", "entity", "sentiment", "text"]

train_df  = pd.read_csv("/content/twitter_training.csv", header=None, names=columns)
val_df  = pd.read_csv("/content/twitter_validation.csv", header=None, names=columns)

Preprocessing

In [ ]:
#Check shape and columns
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTrain columns:", train_df.columns)
print("Validation columns:", val_df.columns)

In [ ]:
print("\nMissing values in TRAIN:")
print(train_df.isna().sum())

print("\nMissing values in VALIDATION:")
print(val_df.isna().sum())

In [ ]:
#Remove null texts

train_df = train_df.dropna(subset=["text"])

print("\nMissing values in TRAIN:")
print(train_df.isna().sum())

print("\nMissing values in VALIDATION:")
print(val_df.isna().sum())

In [ ]:
#Remove irrelevant sentiments

VALID_SENTIMENTS = ["Negative", "Neutral", "Positive"]
train_df = train_df[train_df["sentiment"].isin(VALID_SENTIMENTS)]
val_df   = val_df[val_df["sentiment"].isin(VALID_SENTIMENTS)]

print("\nTrain sentiment distribution after filtering:")
print(train_df["sentiment"].value_counts())

print("\nValidation sentiment distribution after filtering:")
print(val_df["sentiment"].value_counts())

In [ ]:
#Check that there are no duplicates in both datasets

train_texts = set(train_df["text"].astype(str))
val_texts = set(val_df["text"].astype(str))

overlap = train_texts.intersection(val_texts)

print("Number of overlapping texts:", len(overlap))

In [ ]:
# Normalize text to catch near-duplicates
def normalize_text(t):
    return " ".join(str(t).lower().split())

train_df["text_norm"] = train_df["text"].apply(normalize_text)
val_df["text_norm"]   = val_df["text"].apply(normalize_text)

train_norm_set = set(train_df["text_norm"])
val_norm_set   = set(val_df["text_norm"])

overlap_norm = train_norm_set.intersection(val_norm_set)

print("Number of overlapping texts (normalized):", len(overlap_norm))

In [ ]:
#Remove overlaps from training only
train_df = train_df[~train_df["text_norm"].isin(overlap_norm)]

print("Train size after removing overlaps:", len(train_df))
print("Validation size (unchanged):", len(val_df))

In [ ]:
train_df = train_df.drop(columns=["text_norm"])
val_df   = val_df.drop(columns=["text_norm"])

Final Dataset Analysis

In [ ]:
print("\nFinal TRAIN shape:", train_df.shape)
print("Final VALIDATION shape:", val_df.shape)

sns.countplot(x='sentiment', data=train_df)
plt.title("Training Sentiment Distribution")
plt.show()

sns.countplot(x='sentiment', data=val_df)
plt.title("Validation Sentiment Distribution")
plt.show()

In [ ]:
train_df['text_length'] = train_df['text'].apply(len)
sns.histplot(train_df['text_length'])
plt.title("Training Text Length Distribution")
plt.show()

val_df['text_length'] = val_df['text'].apply(len)
sns.histplot(val_df['text_length'])
plt.title("Validation Text Length Distribution")
plt.show()

Train Test Splitting

In [ ]:
#Merge both datasets because the validation set is small

full_df = pd.concat([train_df, val_df], ignore_index=True)

print("Merged dataset shape:", full_df.shape)

In [ ]:
# Ensure no nulls
full_df = full_df.dropna(subset=["text", "sentiment"])

# Normalize sentiment text (safety)
full_df["sentiment"] = (
    full_df["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print(full_df["sentiment"].value_counts())

In [ ]:
X = full_df["text"].values
y = full_df["sentiment"].values

# 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split temp into validation and test (10% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

In [ ]:
#Encode labels

label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc   = label_encoder.transform(y_val)
y_test_enc  = label_encoder.transform(y_test)

print("Label mapping:")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

SVM Model

In [ ]:
#TF-IDF Feature Extraction
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)

In [ ]:
svm_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train_enc)

In [ ]:
y_val_pred_svm = svm_model.predict(X_val_tfidf)

print("SVM Validation Accuracy:", accuracy_score(y_val_enc, y_val_pred_svm))
print("SVM Validation Macro-F1:", f1_score(y_val_enc, y_val_pred_svm, average="macro"))

print("\nValidation Classification Report:\n")
print(classification_report(
    y_val_enc,
    y_val_pred_svm,
    target_names=label_encoder.classes_
))

In [ ]:
cm = confusion_matrix(y_val_enc, y_val_pred_svm)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("SVM Confusion Matrix")
plt.show()

LSTM Model

In [ ]:
#Tokenization and Padding

MAX_VOCAB = 30000
MAX_LEN   = 100   # good default for reviews/comments

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(
    tokenizer.texts_to_sequences(X_train),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_seq = pad_sequences(
    tokenizer.texts_to_sequences(X_val),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_seq = pad_sequences(
    tokenizer.texts_to_sequences(X_test),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [ ]:
#Build LSTM Model Architecture

NUM_CLASSES = len(label_encoder.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_VOCAB,
        output_dim=128,
        input_length=MAX_LEN
    ),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_seq,
    y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=True
)

In [ ]:
y_test_pred_lstm = model.predict(X_test_seq).argmax(axis=1)

print("LSTM Test Accuracy:", accuracy_score(y_test_enc, y_test_pred_lstm))
print("LSTM Test Macro-F1:", f1_score(y_test_enc, y_test_pred_lstm, average="macro"))

print("\nTest Classification Report:\n")
print(classification_report(
    y_test_enc,
    y_test_pred_lstm,
    target_names=label_encoder.classes_
))

In [ ]:
cm = confusion_matrix(y_test_enc, y_test_pred_lstm)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("LSTM Confusion Matrix")
plt.show()

RoBERTa Model

In [ ]:
#Build Hugging Face datasets
train_ds = Dataset.from_dict({"text": X_train, "label": y_train_enc})
val_ds   = Dataset.from_dict({"text": X_val,   "label": y_val_enc})
test_ds  = Dataset.from_dict({"text": X_test,  "label": y_test_enc})

In [ ]:
#Tokenization
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
test_tok  = test_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
#Build model architecture
NUM_CLASSES = len(label_encoder.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

In [ ]:
args = TrainingArguments(
    output_dir="roberta_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
    data_seed=42
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()

In [ ]:
val_metrics = trainer.evaluate(val_tok)
print("RoBERTa Validation:", val_metrics)

In [ ]:
test_pred_roberta = trainer.predict(test_tok)
test_logits = test_pred_roberta.predictions
test_labels = test_pred_roberta.label_ids
test_preds = np.argmax(test_logits, axis=1)

print("RoBERTa Test Accuracy:", accuracy_score(test_labels, test_preds))
print("RoBERTa Test Macro-F1:", f1_score(test_labels, test_preds, average="macro"))

print("\nTest Classification Report:\n")
print(classification_report(
    test_labels,
    test_preds,
    target_names=label_encoder.classes_
))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("RoBERTa Confusion Matrix")
plt.show()